# Demo 03 : Code review

We send a small patch to independent reviewers, filter their findings by severity in code, then ask a judge to merge what is left. A checker written in code confirms that the judge kept the known defects. Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · code-review

Parallel reviewers receive different questions about one patch. Each reports every finding with a severity, and the filter that decides what reaches the judge lives in code.

In [2]:
import operator
from typing import Annotated, Literal

from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel
from typing_extensions import TypedDict

PATCH = '''
def reserve_part(db, part_no, quantity):
    row = db.execute("SELECT stock FROM parts WHERE part_no = ?", (part_no,)).fetchone()
    if row is None:
        return True
    db.execute("UPDATE parts SET stock = stock - ? WHERE part_no = ?", (quantity, part_no))
    return True
'''

class Finding(BaseModel):
    severity: Literal["high", "medium", "low"]
    quote: str
    issue: str

class Review(BaseModel):
    findings: list[Finding]

REVIEWER = ("Review only the assigned concern. Report every issue you find, including low-severity ones "
            "and ones you are unsure about. A later step filters by severity. "
            "For each finding give a severity, a short quote from the patch and the issue.")

class ReviewState(TypedDict):
    patch: str
    findings: Annotated[list[dict], operator.add]
    kept: list[dict]
    accepted: list[dict]

def review(role, concern):
    def run(state):
        result = llm.with_structured_output(Review).invoke([
            SystemMessage(REVIEWER),
            HumanMessage(f"Concern: {concern}\nPatch:\n{state['patch']}")
        ])
        return {"findings": [{"role": role, **f.model_dump()} for f in result.findings]}
    return run

def dispatch(state):
    return [Send(role, {"patch": state["patch"], "findings": [], "kept": [], "accepted": []})
            for role in ["correctness", "database_safety"]]

print(PATCH)


def reserve_part(db, part_no, quantity):
    row = db.execute("SELECT stock FROM parts WHERE part_no = ?", (part_no,)).fetchone()
    if row is None:
        return True
    db.execute("UPDATE parts SET stock = stock - ? WHERE part_no = ?", (quantity, part_no))
    return True



The severity filter is one line of code. We can test it, change the threshold without touching a prompt, and see what it removed. When the reviewers rate every finding high or medium, it passes all of them, and the judge does the narrowing. A prompt that told the reviewers to report only serious issues would hide that decision inside the model.

In [3]:
KEEP = {"high", "medium"}

def severity_filter(state):
    return {"kept": [f for f in state["findings"] if f["severity"] in KEEP]}

class Accepted(BaseModel):
    quote: str
    issue: str

class Judge(BaseModel):
    accepted: list[Accepted]
    rationale: str

def judge(state):
    result = llm.with_structured_output(Judge).invoke(
        "Merge duplicate findings and keep only those supported by the patch. Quote the patch line for each accepted issue. "
        f"Patch:\n{state['patch']}\nFindings:\n{state['kept']}")
    return {"accepted": [a.model_dump() for a in result.accepted]}

graph = StateGraph(ReviewState)
graph.add_node("correctness", review("correctness", "correctness and edge cases"))
graph.add_node("database_safety", review("database_safety", "database safety and transaction behavior"))
graph.add_node("severity_filter", severity_filter)
graph.add_node("judge", judge)
graph.add_conditional_edges(START, dispatch, ["correctness", "database_safety"])
graph.add_edge(["correctness", "database_safety"], "severity_filter")
graph.add_edge("severity_filter", "judge")
graph.add_edge("judge", END)
review_result = graph.compile().invoke({"patch": PATCH, "findings": [], "kept": [], "accepted": []})
for f in review_result["findings"]:
    print(f"{f['role']:16} {f['severity']:6} {f['issue'][:90]}")
print("reported:", len(review_result["findings"]), "kept:", len(review_result["kept"]),
      "accepted by judge:", len(review_result["accepted"]))
for a in review_result["accepted"]:
    print("-", a["issue"][:120])

correctness      high   A missing part is reported as successfully reserved, so callers cannot distinguish an inva
correctness      high   The update does not check that stock is sufficient. Reserving more than available can make
correctness      medium There is no validation that `quantity` is positive. A negative quantity increases stock, a
correctness      low    The function always returns `True` after the update, even if no row was updated (for examp
database_safety  high   A missing part is reported as successfully reserved, so callers can proceed as though stoc
database_safety  high   The stock check and decrement are separate, and the selected stock value is never checked.
database_safety  medium The function does not establish or manage transaction behavior, nor check the update resul
reported: 7 kept: 6 accepted by judge: 3
- A missing part is reported as successfully reserved, so callers cannot distinguish an invalid part number from a success
- The update does not check tha

The patch has two seeded defects : a missing part reports success, and the stock decrement is never checked against the quantity. A checker written in code looks for both in the judge's accepted findings, and rejects the review when one is missing. An empty review is the control, and the checker must reject it.

In [4]:
SEEDED = {
    "missing part reported as success": ["none", "missing", "not found", "nonexist", "does not exist", "unknown part"],
    "stock decrement unchecked": ["negative", "insufficient", "below zero", "enough stock", "available stock", "oversell"],
}

def check(accepted):
    texts = [(a["quote"] + " " + a["issue"]).lower() for a in accepted]
    return {defect: any(m in t for t in texts for m in markers) for defect, markers in SEEDED.items()}

for label, accepted in [("judge", review_result["accepted"]), ("empty control", [])]:
    caught = check(accepted)
    print(f"{label}:", caught, "->", "pass" if all(caught.values()) else "reject")

judge: {'missing part reported as success': True, 'stock decrement unchecked': True} -> pass
empty control: {'missing part reported as success': False, 'stock decrement unchecked': False} -> reject


The checker knows the defects because we seeded them ; on real changes, tests and linters take that role. `no-mistakes` applies this idea to a coding fleet as a pipeline of review, tests, documentation, lint, push, pull request and CI checks.

## Things to try

- Set `KEEP = {"high"}` and see whether a seeded defect is filtered out before the judge
- Tell the reviewers to report only high-severity issues, and compare the number reported